<a href="https://colab.research.google.com/github/Gayathri288/GenAI_LAB_231801039/blob/main/GenAI(ex_6).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ml-latest-small.zip to ml-latest-small.zip


In [ ]:
import zipfile

with zipfile.ZipFile("ml-latest-small.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [ ]:
import pandas as pd

df = pd.read_csv("dataset/ml-latest-small/movies.csv")

print(df.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [ ]:
item_texts = df["title"].astype(str).tolist()

In [ ]:
item_texts = df["title"].astype(str).tolist()

print("Total movies:", len(item_texts))
print("Sample movie:", item_texts[0])

Total movies: 9742
Sample movie: Toy Story (1995)


In [ ]:
!pip install -q faiss-cpu transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import faiss

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

In [ ]:
def encode_texts(texts):

    embeddings = []

    with torch.no_grad():

        for text in texts:

            tokens = tokenizer(
                text,
                return_tensors="pt",
                truncation=True
            ).to(device)

            output = model(**tokens)

            emb = output.last_hidden_state.mean(dim=1)

            embeddings.append(emb.cpu().numpy()[0])

    return np.array(embeddings).astype("float32")


movie_embeddings = encode_texts(item_texts)

In [ ]:
faiss.normalize_L2(movie_embeddings)

dimension = movie_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(movie_embeddings)

print("Total movies indexed:", index.ntotal)

Total movies indexed: 9742


In [ ]:
def get_similar_movies(movie_id, k=5):

    query_vector = movie_embeddings[movie_id:movie_id+1]

    scores, indices = index.search(query_vector, k)

    return scores[0], indices[0]

In [ ]:
query_id = 0

scores, idxs = get_similar_movies(query_id)

print("Query Movie:", item_texts[query_id])

print("\nSimilar Movies:")

for score, idx in zip(scores, idxs):
    print(item_texts[idx], "Score:", score)

Query Movie: Toy Story (1995)

Similar Movies:
Toy Story (1995) Score: 0.99999994
Toy Story 2 (1999) Score: 0.78333867
Toy Story 3 (2010) Score: 0.7601274
Toys (1992) Score: 0.7347038
Toy, The (1982) Score: 0.6891711
